# Line Model — Pretrained Code T5-small model

In [12]:
%tb
import os, json, math, random, glob
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm
from transformers import T5ForConditionalGeneration, AutoTokenizer, get_cosine_schedule_with_warmup
from torch.optim import AdamW

import warnings
warnings.filterwarnings("ignore")

from modules.Plotting import MetricLog, plot_metrics
from modules.EarlyStopping import EarlyStopping
from modules.HandTesting import hand_test_repl
from modules.BestModelSaver import BestModelSaver

WORKDIR = r'C:\Users\Roman\Documents\Projects\code_autocomplete'
print(f"WORKDIR: {WORKDIR}")

LINE_MODEL_NAME = "line_model_T5_small"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] {device}")
print(f"PyTorch версия: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")
print(f"Версия CUDA, под которую собран PyTorch: {torch.version.cuda}")
print(f"Количество GPU: {torch.cuda.device_count()}")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Roman\\Documents\\Projects\\code_autocomplete/plots/line_model_T5_small/line_model_T5_small_ep01.png'

WORKDIR: C:\Users\Roman\Documents\Projects\code_autocomplete
[Device] cuda
PyTorch версия: 2.6.0+cu124
CUDA доступна: True
Версия CUDA, под которую собран PyTorch: 12.4
Количество GPU: 1


## Tokenizer

In [13]:
# character-level BPE-lite

SPECIAL = {"<PAD>": 0, "<UNK>": 1, "<BOS>": 2, "<EOS>": 3}

class CodeTokenizer:
    """
    Simple sub-word tokenizer tailored for Python source code.
    Splits on whitespace/punctuation, keeps indentation tokens,
    and falls back to characters for unknowns.
    """
    PUNCT = set(",;&|~^@#")

    def __init__(self, vocab_size: int = 8000):
        self.vocab_size = vocab_size
        self.token2id: Dict[str, int] = dict(SPECIAL)
        self.id2token: Dict[int, str] = {v: k for k, v in SPECIAL.items()}
        self.built = False

    # ── build ──────────────────────────────────────────────
    def build(self, texts: List[str], min_freq: int = 3):
        freq: Dict[str, int] = defaultdict(int)
        for t in texts:
            for tok in self._raw_split(t):
                freq[tok] += 1
        sorted_tokens = sorted(freq.items(), key=lambda x: -x[1])
        for tok, cnt in sorted_tokens:
            if cnt < min_freq:
                break
            if tok not in self.token2id and len(self.token2id) < self.vocab_size:
                idx = len(self.token2id)
                self.token2id[tok] = idx
                self.id2token[idx] = tok
        # fill remaining slots with single chars
        for c in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789.,_ \t\n":  #for c in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_ \t\n":
            if c not in self.token2id and len(self.token2id) < self.vocab_size:
                idx = len(self.token2id)
                self.token2id[c] = idx
                self.id2token[idx] = c
        self.built = True
        print(f"[Tokenizer] vocab_size={len(self.token2id)}")

    def _raw_split(self, text: str) -> List[str]:
        tokens = []
        for line in text.splitlines(keepends=True):
            # capture leading whitespace as indent token
            stripped = line.lstrip(" \t")
            indent = line[: len(line) - len(stripped)]
            for ch in indent:
                tokens.append(ch)
            # split remainder on punctuation / spaces
            buf = ""
            for ch in stripped:
                if ch in self.PUNCT or ch in " \t\n\r":
                    if ch == "\n" or ch.strip():
                        if buf:
                            tokens.append(buf)
                        tokens.append(ch)
                        continue
                    if buf:
                        buf += ch
                        tokens.append(buf)
                        buf = ""
                        tokens.append("\n")
                else:
                    buf += ch
            if buf:
                tokens.append(buf)
        return tokens

    def encode(self, text: str) -> List[int]:
        ids = [SPECIAL["<BOS>"]]
        for tok in self._raw_split(text):
            if tok in self.token2id:
                ids.append(self.token2id[tok])
            else:
                # char fallback
                for ch in tok:
                    ids.append(self.token2id.get(ch, SPECIAL["<UNK>"]))
        ids.append(SPECIAL["<EOS>"])
        return ids

    def decode(self, ids: List[int]) -> str:
        parts = []
        for i in ids:
            tok = self.id2token.get(i, "")
            if tok in SPECIAL:
                continue
            parts.append(tok)
        return "".join(parts)

    def save(self, path: str):
        with open(path, "w") as f:
            json.dump({"token2id": self.token2id}, f)

    @classmethod
    def load(cls, path: str) -> "CodeTokenizer":
        with open(path) as f:
            d = json.load(f)
        obj = cls()
        obj.token2id = {k: int(v) for k, v in d["token2id"].items()}
        obj.id2token = {v: k for k, v in obj.token2id.items()}
        obj.built = True
        return obj

    @property
    def pad_id(self):  return SPECIAL["<PAD>"]
    @property
    def eos_id(self):  return SPECIAL["<EOS>"]
    @property
    def bos_id(self):  return SPECIAL["<BOS>"]
    @property
    def vocab(self):   return len(self.token2id)

## Datasets

In [14]:

def load_files(data_dir: str, max_files: int = 0) -> List[str]:
    """Load .py / .txt files from a directory tree."""
    patterns = ["**/*.py", "**/*.txt"]
    files = []
    for pat in patterns:
        files.extend(glob.glob(os.path.join(data_dir, pat), recursive=True))
    if max_files:
        files = files[:max_files]
    texts = []
    for fp in files:
        try:
            texts.append(Path(fp).read_text(errors="replace"))
        except Exception:
            pass
    print(f"[Data] loaded {len(texts)} files from {data_dir}")
    return texts



class T5LineDataset(Dataset):
    """
    Each sample: tokenise the prefix with AutoTokenizer → input_ids
                 tokenise the suffix                        → labels
    Padding and label-masking are handled in the collator below.
    """
    def __init__(self, texts: List[str], hf_tokenizer: AutoTokenizer,
                 max_prefix: int = 96, max_suffix: int = 64):
        self.tok = hf_tokenizer
        self.max_prefix = max_prefix
        self.max_suffix = max_suffix
        self.samples: List[Tuple[str, str]] = []

        for text in texts:
            for line in text.splitlines():
                line = line.rstrip()
                if len(line.strip()) < 10:
                    continue
                # split at 30–70 % of the line (your "root cause 2" fix)
                words = line.split()
                if len(words) < 3:
                    continue
                cut = random.randint(
                    max(1, int(len(words) * 0.3)),
                    max(2, int(len(words) * 0.7)),
                )
                prefix = " ".join(words[:cut])
                suffix = " ".join(words[cut:])
                self.samples.append((prefix, suffix))

        print(f"[T5LineDataset] {len(self.samples)} samples")

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        prefix, suffix = self.samples[i]
        enc = self.tok(
            prefix,
            max_length=self.max_prefix,
            truncation=True,
            padding=False,
            return_tensors="pt",
        )
        dec = self.tok(
            suffix,
            max_length=self.max_suffix,
            truncation=True,
            padding=False,
            return_tensors="pt",
        )
        return (
            enc["input_ids"].squeeze(0),
            enc["attention_mask"].squeeze(0),
            dec["input_ids"].squeeze(0),
        )



def collate_t5(batch, pad_id: int):
    """Pad input_ids, attention_mask, and labels in a single pass."""
    src_ids, src_masks, lbl_ids = zip(*batch)

    max_src = max(t.size(0) for t in src_ids)
    max_lbl = max(t.size(0) for t in lbl_ids)

    B = len(batch)
    SRC  = torch.full((B, max_src), pad_id, dtype=torch.long)
    MASK = torch.zeros((B, max_src), dtype=torch.long)
    LBL  = torch.full((B, max_lbl), -100,   dtype=torch.long)   # -100 = ignored by T5 loss

    for i, (s, m, l) in enumerate(zip(src_ids, src_masks, lbl_ids)):
        SRC[i,  :s.size(0)] = s
        MASK[i, :m.size(0)] = m
        LBL[i,  :l.size(0)] = l

    return SRC, MASK, LBL

## Models

In [15]:
@dataclass
class ModelCfg:
    vocab: int = 8000
    d_model: int = 256
    n_heads: int = 8
    n_layers: int = 4
    d_ff: int = 1024
    max_len: int = 256
    dropout: float = 0.1


## Training loop

In [31]:
def _clip_norm(model: nn.Module, max_norm: float = 1.0) -> float:
    return nn.utils.clip_grad_norm_(model.parameters(), max_norm).item()


def train_line_model(
    model:           T5ForConditionalGeneration,
    hf_tok:          AutoTokenizer,
    train_dl:        DataLoader,
    val_dl:          DataLoader,
    epochs:          int,
    lr:              float,
    device:          torch.device,
    saver:           BestModelSaver,
    log:             MetricLog,
    plot_dir:        str,
    label_smoothing: float = 0.1,
    warmup_frac:     float = 0.05,
    patience:        int   = 3,
    use_amp:         bool  = True,
):
    tqdm.write(f"[Line] DataLoader — {len(train_dl)} train batches, "
               f"{len(val_dl)} val batches")

    opt          = AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    total_steps  = len(train_dl) * epochs
    warmup_steps = max(1, int(total_steps * warmup_frac))
    sched        = get_cosine_schedule_with_warmup(opt, warmup_steps, total_steps)

    # T5 uses bfloat16 better than float16 — use bf16 if available, else fp16
    amp_enabled = use_amp and device.type == "cuda"
    amp_dtype   = torch.bfloat16 if (amp_enabled and torch.cuda.is_bf16_supported()) else torch.float16
    scaler      = GradScaler("cuda", enabled=(amp_enabled and amp_dtype == torch.float16))
    tqdm.write(f"[AMP] enabled={amp_enabled}, dtype={amp_dtype}")

    stopper = EarlyStopping(patience=patience)

    for ep in range(1, epochs + 1):
        # ── TRAIN ────────────────────────────────────────────────────────────
        model.train()
        t_loss = t_acc = t_steps = 0
        gn = 0.0

        batch_bar = tqdm(train_dl,
                         desc=f"[Line] Epoch {ep}/{epochs} train",
                         leave=False, unit="batch")

        for src, mask, lbl in batch_bar:
            src, mask, lbl = src.to(device, non_blocking=True), mask.to(device, non_blocking=True), lbl.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)

            with autocast("cuda", dtype=amp_dtype, enabled=amp_enabled):
                # T5 still does the label right-shift internally; we just override the loss
                out    = model(input_ids=src, attention_mask=mask, labels=lbl)
                logits = out.logits   # (B, T_dec, V) — aligned with lbl
                loss   = F.cross_entropy(
                    logits.reshape(-1, logits.size(-1)),
                    lbl.reshape(-1),
                    ignore_index=-100,
                    label_smoothing=label_smoothing,
                )

            if amp_dtype == torch.float16:
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                gn = _clip_norm(model)
                scaler.step(opt)
                scaler.update()
            else:
                # bf16 doesn't need GradScaler
                loss.backward()
                gn = _clip_norm(model)
                opt.step()
            sched.step()

            with torch.no_grad():
                preds = logits.argmax(-1)
                valid = (lbl != -100)
                acc   = (preds[valid] == lbl[valid]).float().mean().item() if valid.any() else 0.0

            t_loss  += loss.item()
            t_acc   += acc
            t_steps += 1

            batch_bar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{acc:.3f}",
                                  lr=f"{opt.param_groups[0]['lr']:.2e}")

        tl = t_loss / t_steps
        ta = t_acc  / t_steps

        # ── VAL ──────────────────────────────────────────────────────────────
        model.eval()
        v_loss = v_steps = 0
        with torch.no_grad():
            for src, mask, lbl in tqdm(val_dl,
                                       desc=f"[Line] Epoch {ep}/{epochs} val  ",
                                       leave=False, unit="batch"):
                src, mask, lbl = src.to(device, non_blocking=True), mask.to(device, non_blocking=True), lbl.to(device, non_blocking=True)
                with autocast("cuda", dtype=amp_dtype, enabled=amp_enabled):
                    out    = model(input_ids=src, attention_mask=mask, labels=lbl)
                    logits = out.logits
                    loss   = F.cross_entropy(
                        logits.reshape(-1, logits.size(-1)),
                        lbl.reshape(-1),
                        ignore_index=-100,
                        label_smoothing=label_smoothing,
                    )
                v_loss  += loss.item()
                v_steps += 1

        vl = v_loss / v_steps if v_steps else tl

        log.append(
            train_loss=tl, val_loss=vl,
            train_ppl=math.exp(min(tl, 20)), val_ppl=math.exp(min(vl, 20)),
            lr=opt.param_groups[0]["lr"],
            token_acc=ta, grad_norm=gn,
        )
        tqdm.write(
            f"[Line  ep {ep:3d}] train_loss={tl:.4f}  val_loss={vl:.4f}"
            f"  ppl={math.exp(min(vl,20)):.1f}  acc={ta:.3f}"
            f"  lr={opt.param_groups[0]['lr']:.2e}"
        )

        saver.save(model, vl, ep)
        if ep % max(1, epochs // 5) == 0 or ep == epochs:
            plot_metrics(log, f"{LINE_MODEL_NAME.replace('_', ' ')} — Epoch {ep}",
                         f"{plot_dir}/{LINE_MODEL_NAME}_ep{ep:02d}.png")

        if stopper(vl):
            tqdm.write(f"[Early stop] val_loss did not improve for {stopper.patience} epochs — stopping.")
            break

    plot_metrics(log, f"{LINE_MODEL_NAME.replace('_', ' ')} — Final",
                 f"{plot_dir}/{LINE_MODEL_NAME}_final.png")

## Main

In [32]:
class Arguments():
    def __init__(self, data_dir: str = f"{WORKDIR}/Clean_Dataset", ckpt_dir: str = f"{WORKDIR}/checkpoints/{LINE_MODEL_NAME}",
                    plot_dir: str = F"{WORKDIR}/plots/{LINE_MODEL_NAME}", tokenizer: str = "tokenizer.json", 
                    epochs: int = 5, batch: int = 32, lr: float = 5e-4,
                    ctx: int = 128, d_model: int = 256, n_layers: int = 4,
                    n_heads: int = 8, vocab_size: int = 000, max_files: int = 0,
                    val_split: float = 0.1, seed: int = 42, for_usage: bool = False,
                    skip_token: bool = False, skip_line: bool = False, test: bool = False):
        self.data_dir = data_dir
        self.ckpt_dir = ckpt_dir
        self.plot_dir = plot_dir
        self.tokenizer = tokenizer
        self.epochs = epochs
        self.batch = batch
        self.lr = lr
        self.ctx = ctx
        self.d_model = d_model
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.vocab_size = vocab_size
        self.max_files = max_files
        self.val_split = val_split
        self.seed = seed
        self.skip_token = skip_token
        self.skip_line = skip_line
        self.test = test
        self.for_usage = for_usage


def main():
    args = Arguments(epochs=10)
    # args = Arguments(max_files=10)
    # args = Arguments(skip_line=True, max_files=100, epochs=1)
    # args = Arguments(max_files=100, epochs=2, skip_token=True, vocab_size=160000)
    # args = Arguments(skip_token=True)
    # args = Arguments(test=True)
    # args = Arguments(for_usage==True)
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)

    os.makedirs(args.ckpt_dir, exist_ok=True)
    os.makedirs(args.plot_dir,  exist_ok=True)

    # ── tokenizer ────────────────────────────────────────────
    if os.path.exists(args.tokenizer):
        print(f"[Tokenizer] loading {args.tokenizer}")
        tokenizer = CodeTokenizer.load(args.tokenizer)
    else:
        print("[Tokenizer] building from data …")
        texts = load_files(args.data_dir, args.max_files)
        tokenizer = CodeTokenizer(vocab_size=args.vocab_size)
        tokenizer.build(texts)
        tokenizer.save(args.tokenizer)

    torch.serialization.add_safe_globals([ModelCfg])

    HF_MODEL = "Salesforce/codet5-small"
    hf_tok = AutoTokenizer.from_pretrained(HF_MODEL)

    # ── test-only mode ───────────────────────────────────────
    if args.test:
        lm = T5ForConditionalGeneration.from_pretrained(HF_MODEL).to(device)
        line_paths = sorted(glob.glob(str(Path(args.ckpt_dir) / LINE_MODEL_NAME / f"{LINE_MODEL_NAME}_*.pt")))
        if line_paths:
            ck = torch.load(line_paths[0], map_location=device, weights_only=False)
            lm.load_state_dict(ck["model_state"])
            print(f"[Loaded] line model from {line_paths[0]}")
    
        hand_test_repl(None, lm, None, hf_tok, device)
        return
    

    # ── load data ────────────────────────────────────────────
    print("[Loading] Started loading")
    texts = load_files(args.data_dir, args.max_files)
    if not texts:
        print("[ERROR] no data files found. Please put .py files in --data_dir")
        return
    print("[Loading] Ended loading")

    random.shuffle(texts)
    split = max(1, int(len(texts) * (1 - args.val_split)))
    tr_txt = texts[:split]
    va_txt = texts[split:]

    # ── LINE MODEL ───────────────────────────────────────────
    if not args.skip_line:
        print("  Prepairing LINE model")

        line_model = T5ForConditionalGeneration.from_pretrained(HF_MODEL).to(device)
        
        collate    = lambda b: collate_t5(b, hf_tok.pad_token_id)
        tr_line_ds = T5LineDataset(tr_txt, hf_tok)
        va_line_ds = T5LineDataset(va_txt, hf_tok)
        tr_line_dl = DataLoader(tr_line_ds, args.batch, shuffle=True,
                                collate_fn=collate, num_workers=0, pin_memory=True)
        va_line_dl = DataLoader(va_line_ds, args.batch, shuffle=False,
                                collate_fn=collate, num_workers=0, pin_memory=True)
        
        n_params = sum(p.numel() for p in line_model.parameters() if p.requires_grad)
        print(f"[Line  Model] {n_params/1e6:.1f}M parameters (codet5-small)")
        
        line_saver = BestModelSaver(args.ckpt_dir, LINE_MODEL_NAME, from_hf=True)
        line_log   = MetricLog()
        train_line_model(line_model, hf_tok, tr_line_dl, va_line_dl,
                         args.epochs, args.lr, device, line_saver, line_log, args.plot_dir)

    # ── interactive test ─────────────────────────────────────
    hand_test_repl(None, line_model, tokenizer, hf_tok, device)


main()

[Tokenizer] loading tokenizer.json
[Loading] Started loading
[Data] loaded 12110 files from C:\Users\Roman\Documents\Projects\code_autocomplete/Clean_Dataset
[Loading] Ended loading
  Prepairing LINE model
[T5LineDataset] 676196 samples
[T5LineDataset] 73886 samples
[Line  Model] 60.5M parameters (codet5-small)
[Line] DataLoader — 21132 train batches, 2309 val batches


[Line  ep   1] train_loss=2.3712  val_loss=2.1274  ppl=8.4  acc=0.566  lr=4.88e-04
[Saver] saved ckpt: C:\Users\Roman\Documents\Projects\code_autocomplete\checkpoints\line_model_T5_small\line_model_T5_small_ep001_loss2.1274.pt  (val_loss=2.1274)


[Line  ep   2] train_loss=2.0249  val_loss=2.0385  ppl=7.7  acc=0.611  lr=4.55e-04
[Saver] saved ckpt: C:\Users\Roman\Documents\Projects\code_autocomplete\checkpoints\line_model_T5_small\line_model_T5_small_ep002_loss2.0385.pt  (val_loss=2.0385)
[Plot] saved → C:\Users\Roman\Documents\Projects\code_autocomplete/plots/line_model_T5_small/line_model_T5_small_ep02.png


[Line  ep   3] train_loss=1.8620  val_loss=1.9961  ppl=7.4  acc=0.634  lr=4.02e-04
[Saver] saved ckpt: C:\Users\Roman\Documents\Projects\code_autocomplete\checkpoints\line_model_T5_small\line_model_T5_small_ep003_loss1.9961.pt  (val_loss=1.9961)


[Line  ep   4] train_loss=1.7378  val_loss=1.9549  ppl=7.1  acc=0.652  lr=3.36e-04
[Saver] removed old ckpt: C:\Users\Roman\Documents\Projects\code_autocomplete\checkpoints\line_model_T5_small\line_model_T5_small_ep001_loss2.1274.pt
[Saver] saved ckpt: C:\Users\Roman\Documents\Projects\code_autocomplete\checkpoints\line_model_T5_small\line_model_T5_small_ep004_loss1.9549.pt  (val_loss=1.9549)
[Plot] saved → C:\Users\Roman\Documents\Projects\code_autocomplete/plots/line_model_T5_small/line_model_T5_small_ep04.png


...

[Line  ep   6] train_loss=1.5262  val_loss=1.9149  ppl=6.8  acc=0.684  lr=1.89e-04
[Saver] removed old ckpt: C:\Users\Roman\Documents\Projects\code_autocomplete\checkpoints\line_model_T5_small\line_model_T5_small_ep003_loss1.9961.pt
[Saver] saved ckpt: C:\Users\Roman\Documents\Projects\code_autocomplete\checkpoints\line_model_T5_small\line_model_T5_small_ep006_loss1.9149.pt  (val_loss=1.9149)
[Plot] saved → C:\Users\Roman\Documents\Projects\code_autocomplete/plots/line_model_T5_small/line_model_T5_small_ep06.png


[Line  ep   7] train_loss=1.4287  val_loss=1.8989  ppl=6.7  acc=0.700  lr=1.23e-04
[Saver] removed old ckpt: C:\Users\Roman\Documents\Projects\code_autocomplete\checkpoints\line_model_T5_small\line_model_T5_small_ep004_loss1.9549.pt
[Saver] saved ckpt: C:\Users\Roman\Documents\Projects\code_autocomplete\checkpoints\line_model_T5_small\line_model_T5_small_ep007_loss1.8989.pt  (val_loss=1.8989)


[Line  ep   8] train_loss=1.3401  val_loss=1.8931  ppl=6.6  acc=0.715  lr=7.04e-05
[Saver] removed old ckpt: C:\Users\Roman\Documents\Projects\code_autocomplete\checkpoints\line_model_T5_small\line_model_T5_small_ep005_loss1.9360.pt
[Saver] saved ckpt: C:\Users\Roman\Documents\Projects\code_autocomplete\checkpoints\line_model_T5_small\line_model_T5_small_ep008_loss1.8931.pt  (val_loss=1.8931)
[Plot] saved → C:\Users\Roman\Documents\Projects\code_autocomplete/plots/line_model_T5_small/line_model_T5_small_ep08.png


[Line  ep   9] train_loss=1.2680  val_loss=1.8923  ppl=6.6  acc=0.728  lr=3.66e-05
[Saver] removed old ckpt: C:\Users\Roman\Documents\Projects\code_autocomplete\checkpoints\line_model_T5_small\line_model_T5_small_ep006_loss1.9149.pt
[Saver] saved ckpt: C:\Users\Roman\Documents\Projects\code_autocomplete\checkpoints\line_model_T5_small\line_model_T5_small_ep009_loss1.8923.pt  (val_loss=1.8923)


KeyboardInterrupt: 

### Evaluation

In [ ]:
"""
evaluate.py — Offline evaluation of trained autocomplete models.

Metrics computed:
  Token model : top-1 / top-5 accuracy, perplexity, mean reciprocal rank
  Line  model : exact-match@1, prefix-match, BLEU-4, chrF, perplexity
  Both        : latency (ms / sample)

Results are saved to JSON and a summary PNG dashboard.
"""

import os, json, time, glob, math, argparse
from pathlib import Path
from typing import List, Tuple, Optional

import torch
import torch.nn.functional as F
import numpy as np

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# import models from train.py (same directory)
import sys
sys.path.insert(0, os.path.dirname(__file__))


# ─────────────────────────────────────────────────────────────
# BLEU / chrF helpers (no NLTK dependency)
# ─────────────────────────────────────────────────────────────

def ngrams(seq: list, n: int) -> dict:
    counts = {}
    for i in range(len(seq) - n + 1):
        g = tuple(seq[i: i+n])
        counts[g] = counts.get(g, 0) + 1
    return counts


def bleu4(ref: list, hyp: list) -> float:
    if len(hyp) == 0:
        return 0.0
    bp = min(1.0, math.exp(1 - len(ref) / max(1, len(hyp))))
    score = 0.0
    for n in range(1, 5):
        r_ng = ngrams(ref, n)
        h_ng = ngrams(hyp, n)
        clip = sum(min(h_ng[g], r_ng.get(g, 0)) for g in h_ng)
        tot  = max(1, sum(h_ng.values()))
        if clip == 0:
            return 0.0
        score += math.log(clip / tot)
    return bp * math.exp(score / 4)


def chrf(ref: str, hyp: str, n: int = 6) -> float:
    def char_ngrams(s, n):
        return ngrams(list(s), n)
    scores = []
    for i in range(1, n + 1):
        r = char_ngrams(ref, i)
        h = char_ngrams(hyp, i)
        prec = sum(min(h[g], r.get(g, 0)) for g in h) / max(1, sum(h.values()))
        rec  = sum(min(r[g], h.get(g, 0)) for g in r) / max(1, sum(r.values()))
        f    = 2 * prec * rec / max(1e-9, prec + rec)
        scores.append(f)
    return float(np.mean(scores)) if scores else 0.0


# ─────────────────────────────────────────────────────────────
# Evaluation routines
# ─────────────────────────────────────────────────────────────

# @torch.no_grad()
# def eval_token_model(model: TokenModel, dataset: TokenDataset,
#                      device: torch.device, n_samples: int = 2000):
#     model.eval()
#     crit = torch.nn.CrossEntropyLoss(ignore_index=SPECIAL["<PAD>"], reduction="sum")
#     total_loss = total_tokens = 0
#     top1_correct = top5_correct = 0
#     mrr_sum = 0.0
#     latencies = []

#     indices = list(range(min(n_samples, len(dataset))))
#     np.random.shuffle(indices)

#     for idx in indices:
#         x, y = dataset[idx]
#         x = x.unsqueeze(0).to(device)
#         y = y.unsqueeze(0).to(device)

#         t0 = time.perf_counter()
#         logits = model(x)
#         latencies.append((time.perf_counter() - t0) * 1000)

#         loss = crit(logits.view(-1, logits.size(-1)), y.view(-1))
#         n_tok = (y != SPECIAL["<PAD>"]).sum().item()
#         total_loss   += loss.item()
#         total_tokens += n_tok

#         # last-position metrics
#         last_logit = logits[0, -1]
#         true_id    = y[0, -1].item()
#         if true_id == SPECIAL["<PAD>"]:
#             continue
#         sorted_ids = last_logit.argsort(descending=True).tolist()
#         rank = sorted_ids.index(true_id) + 1 if true_id in sorted_ids else len(sorted_ids)
#         top1_correct += (rank == 1)
#         top5_correct += (rank <= 5)
#         mrr_sum      += 1.0 / rank

#     n = len(indices)
#     return {
#         "perplexity":  math.exp(min(total_loss / max(1, total_tokens), 20)),
#         "top1_acc":    top1_correct / n,
#         "top5_acc":    top5_correct / n,
#         "mrr":         mrr_sum / n,
#         "latency_ms":  float(np.mean(latencies)),
#     }


@torch.no_grad()
def eval_line_model(model: LineModel, dataset: LineDataset,
                    tokenizer: CodeTokenizer, device: torch.device,
                    n_samples: int = 500):
    model.eval()
    exact = prefix20 = 0
    bleu_scores = []
    chrf_scores = []
    latencies   = []

    indices = list(range(min(n_samples, len(dataset))))
    np.random.shuffle(indices)

    for idx in indices:
        prefix_ids, target_ids = dataset[idx]
        # strip EOS from target for comparison
        target_ids = [i for i in target_ids if i != SPECIAL["<EOS>"]]

        t0 = time.perf_counter()
        hyp_ids = model.generate(prefix_ids, max_new=64,
                                 temperature=1.0, top_k=1,
                                 tokenizer=tokenizer)
        latencies.append((time.perf_counter() - t0) * 1000)

        exact    += (hyp_ids == target_ids)
        prefix20 += (hyp_ids[:20] == target_ids[:20])
        bleu_scores.append(bleu4(target_ids, hyp_ids))
        ref_str = tokenizer.decode(target_ids)
        hyp_str = tokenizer.decode(hyp_ids)
        chrf_scores.append(chrf(ref_str, hyp_str))

    n = len(indices)
    return {
        "exact_match":    exact / n,
        "prefix20_match": prefix20 / n,
        "bleu4":          float(np.mean(bleu_scores)),
        "chrf":           float(np.mean(chrf_scores)),
        "latency_ms":     float(np.mean(latencies)),
    }


# ─────────────────────────────────────────────────────────────
# Dashboard plot
# ─────────────────────────────────────────────────────────────

def plot_eval(tok_res: dict, line_res: dict, out_path: str):
    DARK  = "#0d1117"; MID = "#161b22"; GRID = "#21262d"
    BLUE  = "#58a6ff"; GREEN = "#3fb950"; ORG = "#ffa657"; TXT = "#c9d1d9"

    plt.rcParams.update({
        "axes.facecolor": MID, "axes.edgecolor": GRID,
        "axes.labelcolor": TXT, "xtick.color": TXT,
        "ytick.color": TXT, "text.color": TXT, "grid.color": GRID,
    })

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=DARK)

    # Token model bar chart
    ax = axes[0]
    metrics = ["top1_acc", "top5_acc", "mrr"]
    values  = [tok_res[m] for m in metrics]
    labels  = ["Top-1 Acc", "Top-5 Acc", "MRR"]
    bars = ax.bar(labels, values, color=[BLUE, GREEN, ORG], width=0.5, zorder=3)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.01, f"{val:.3f}",
                ha="center", va="bottom", fontsize=10, color=TXT)
    ax.set_ylim(0, 1.05)
    ax.set_title(f"Token Model\nPerplexity: {tok_res['perplexity']:.1f}  "
                 f"Latency: {tok_res['latency_ms']:.1f}ms",
                 fontsize=10, color=BLUE, pad=8)
    ax.grid(True, axis="y", lw=0.5, zorder=0)

    # Line model bar chart
    ax = axes[1]
    metrics2 = ["exact_match", "prefix20_match", "bleu4", "chrf"]
    values2  = [line_res[m] for m in metrics2]
    labels2  = ["Exact\nMatch", "Prefix-20\nMatch", "BLEU-4", "chrF"]
    bars2 = ax.bar(labels2, values2, color=[BLUE, GREEN, ORG, "#d2a8ff"], width=0.5, zorder=3)
    for bar, val in zip(bars2, values2):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.01, f"{val:.3f}",
                ha="center", va="bottom", fontsize=10, color=TXT)
    ax.set_ylim(0, 1.05)
    ax.set_title(f"Line Model\nLatency: {line_res['latency_ms']:.1f}ms",
                 fontsize=10, color=BLUE, pad=8)
    ax.grid(True, axis="y", lw=0.5, zorder=0)

    fig.suptitle("Evaluation Dashboard", fontsize=14, color=BLUE)
    plt.tight_layout()
    plt.savefig(out_path, dpi=130, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"[Eval] plot → {out_path}")


# ─────────────────────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────────────────────

def main():
    p = argparse.ArgumentParser()
    p.add_argument("--data_dir",   default="data/val")
    p.add_argument("--ckpt_dir",   default="checkpoints")
    p.add_argument("--tokenizer",  default="tokenizer.json")
    p.add_argument("--out_dir",    default="eval_results")
    p.add_argument("--n_tok",      type=int, default=2000)
    p.add_argument("--n_line",     type=int, default=500)
    p.add_argument("--ctx",        type=int, default=128)
    args = p.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)
    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = CodeTokenizer.load(args.tokenizer)

    # ── load best checkpoints ────────────────────────────────
    def load_best(prefix: str, model_cls, cfg):
        paths = sorted(glob.glob(str(Path(args.ckpt_dir) / f"{prefix}_*.pt")))
        if not paths:
            print(f"[Eval] No {prefix} checkpoint found in {args.ckpt_dir}")
            return None
        ck = torch.load(paths[0], map_location=device)
        m  = model_cls(cfg).to(device)
        m.load_state_dict(ck["model_state"])
        print(f"[Eval] loaded {prefix} from {paths[0]}  (val_loss={ck['val_loss']:.4f})")
        return m

    cfg = ModelCfg(vocab=tokenizer.vocab, d_model=256, n_heads=8,
                   n_layers=4, d_ff=1024, max_len=args.ctx+32)

    tok_model  = load_best(TOKEN_MODEL_NAME, TokenModel, cfg)
    line_model = load_best(LINE_MODEL_NAME, LineModel, cfg)

    texts = load_files(args.data_dir)
    if not texts:
        print("[Eval] no data found — run prepare_data.py first"); return

    all_ids = []
    for t in texts:
        all_ids.extend(tokenizer.encode(t))

    results = {}

    if tok_model:
        print("[Eval] evaluating Token model …")
        ds  = TokenDataset(all_ids, args.ctx)
        res = eval_token_model(tok_model, ds, device, args.n_tok)
        results["token"] = res
        print(json.dumps(res, indent=2))

    if line_model:
        print("[Eval] evaluating Line model …")
        ds  = LineDataset(texts, tokenizer)
        res = eval_line_model(line_model, ds, tokenizer, device, args.n_line)
        results["line"] = res
        print(json.dumps(res, indent=2))

    out_json = os.path.join(args.out_dir, "eval_results.json")
    with open(out_json, "w") as f:
        json.dump(results, f, indent=2)
    print(f"[Eval] results → {out_json}")

    if tok_model and line_model:
        plot_eval(results["token"], results["line"],
                  os.path.join(args.out_dir, "eval_dashboard.png"))


main()